Install Dependencies

In [1]:
!pip install openai pillow pypdf
!pip install --upgrade openai

Structure Output into JSON

In [2]:
from openai import OpenAI
from IPython.display import Image, display
import base64
import mimetypes
import json

client = OpenAI()

def get_billing_details_normal(image_path):
    # Read image and encode
    with open(image_path, "rb") as f:
        img_bytes = f.read()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    # Guess MIME type
    mime_type, _ = mimetypes.guess_type(image_path)

    # Send to OpenAI
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": """Extract billing details from this image.
Return JSON with fields:
- invoice_number
- date
- customer_name
- items (list of {"description","quantity","unit_price","total"})
- grand_total"""
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{img_b64}"
                        }
                    },
                ],
            }
        ]
    )

    # Parse JSON safely
    raw_output = response.choices[0].message.content.strip()
    try:
        # Remove code fences if present
        if raw_output.startswith("```"):
            raw_output = raw_output.split("```")[1].strip()
        parsed_data = json.loads(raw_output)
    except json.JSONDecodeError:
        parsed_data = {"error": "Could not parse JSON", "raw_output": raw_output}

    return parsed_data


Load and Prepare the Image

In [3]:
def calculate_total(billing_json):
    return sum(item["quantity"] * item["price"] for item in billing_json["items"])

Send Image to OpenAI for Parsing

In [4]:
def get_billing_details_extract_custom(image_path):
   
    with open(image_path, "rb") as f:
        img_bytes = f.read()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    mime_type, encoding = mimetypes.guess_type(image_path)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=300,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                         "text": """Extract billing details in JSON format:
                        {
                          "invoice_number": "",
                          "date": "",
                          "total": "",
                          "line items": "",
                          "items": [{"description": "", "quantity": "", "price": ""}]
                        }"""
                    },
                    # The content type should be "image_url" to use gpt-4-turbo's vision capabilities
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/{mime_type};base64,{img_b64}"
                        }
                    },
                ],
             }
            ]
        )
    
    billing_json = response.choices[0].message.content
    print(billing_json)

In [5]:
def get_billing_custom(image_path):
    img = Image(url=filename)
    display(img)
    print("_____________________Parse Image________________________")
    get_billing_details_normal(filename)
    print("_____________________Parse Image into formattted_________________________")
    get_billing_details_extract_custom(filename)

In [6]:
print("_____________________Sample Invoice Details________________________")
filename="sampleinvoice.png"
get_billing_custom(filename)
print("_____________________Handwritten Invoice Details________________________")
filename="handwritteninvoice.png"
get_billing_custom(filename)

_____________________Sample Invoice Details________________________


_____________________Parse Image________________________
_____________________Parse Image into formattted_________________________
Here are the extracted billing details in JSON format:

```json
{
  "invoice_number": "INV-10012",
  "date": "26/03/2021",
  "total": "$1,699.48",
  "line items": "3",
  "items": [
    {
      "description": "Services",
      "quantity": "10",
      "price": "$550.00"
    },
    {
      "description": "Consulting",
      "quantity": "15",
      "price": "$1,125.00"
    },
    {
      "description": "Materials",
      "quantity": "1",
      "price": "$123.39"
    }
  ]
}
```
_____________________Handwritten Invoice Details________________________


_____________________Parse Image________________________
_____________________Parse Image into formattted_________________________
```json
{
  "invoice_number": "",
  "date": "May 1814",
  "total": "£327 15s 2d",
  "line items": [
    {"description": "Jaconet", "quantity": "67", "price": "£266"},
    {"description": "White Mull", "quantity": "90", "price": ""},
    {"description": "Fang White Muslin", "quantity": "26", "price": ""},
    {"description": "Jonguil D^2", "quantity": "28", "price": ""},
    {"description": "Coquelicot D^2", "quantity": "55", "price": ""}
  ],
  "items": [
    {"description": "Case", "quantity": "1", "price": "£2 5s"},
    {"description": "Insurance 2 Guineas per cent", "quantity": "1", "price": "£51 9s 2d"},
    {"description": "Carriage to Hull", "quantity": "1", "price": "£4 8s"},
    {"description": "Shipping", "quantity": "1", "price": "£18 6s"},
    {"description": "Duty", "quantity": "1", "price": "£31 7s"}
  ]
}
```


In [7]:
from pypdf import PdfReader

def extract_pdf_text(file_path):
    reader = PdfReader(file_path)
    text_content = ""

    for page in reader.pages:
        text_content += page.extract_text() + "\n"

    return text_content

pdf_text = extract_pdf_text("wordpress-pdf-invoice-plugin-sample.pdf")
print(pdf_text)


Invoice
Payment is due within 30 days from date of invoice. Late payment is subject to fees of 5% per month.
Thanks for choosing DEMO - Sliced Invoices | admin@slicedinvoices.com
Page 1/1
From:
DEMO - Sliced Invoices
Suite 5A-1204
123 Somewhere Street
Your City AZ 12345
admin@slicedinvoices.com
Invoice Number INV-3337
Order Number 12345
Invoice Date January 25, 2016
Due Date January 31, 2016
Total Due $93.50
To:
Test Business
123 Somewhere St
Melbourne, VIC 3000
test@test.com
Hrs/Qty Service Rate/Price Adjust Sub Total
1.00 Web Design
This is a sample description... $85.00 0.00% $85.00
Sub Total $85.00
Tax $8.50
Total $93.50
ANZ Bank
ACC # 1234 1234
BSB # 4321 432 Paid



In [8]:
prompt = f"""
You are a bill parser. Extract structured data from the following bill text:
{pdf_text}

Return JSON with fields:
- invoice_number
- date
- customer_name
- items (list of {{"description","quantity","unit_price","total"}})
- grand_total
"""

response = client.chat.completions.create(
    model="gpt-4.1",
    messages=[{"role":"user","content":prompt}],
    temperature=0
)

parsed_json = response.choices[0].message.content
print(parsed_json)

```json
{
  "invoice_number": "INV-3337",
  "date": "January 25, 2016",
  "customer_name": "Test Business",
  "items": [
    {
      "description": "Web Design\nThis is a sample description...",
      "quantity": 1.00,
      "unit_price": 85.00,
      "total": 85.00
    }
  ],
  "grand_total": 93.50
}
```
